# scChromatic end-to-end demonstration on a public human pancreas dataset

## Dataset choice

This notebook uses the **Segerstolpe human pancreas Smart-seq2 dataset** (ArrayExpress accession **E-MTAB-5061**) through `scRNAseq::SegerstolpePancreasData()`. The public dataset contains gene-expression measurements and author-provided metadata for 3,514 cells from 10 human donors, including healthy and type-2-diabetes conditions and multiple endocrine, exocrine, stromal and immune cell types.

One dataset therefore supplies every input needed for a coherent scChromatic demonstration:

- cell-type labels for persistent categorical maps;
- donor and disease metadata for subset/reorder stress tests;
- multiple donors for sample-balanced relationship affinities;
- expression counts for normalization, PCA and continuous marker display;
- a biological parent-child structure for hierarchy-aware colors.

**Sources:** [original study](https://doi.org/10.1016/j.cmet.2016.08.020), [E-MTAB-5061](https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-5061), [Bioconductor workflow](https://bioconductor.org/books/release/OSCA.workflows/segerstolpe-human-pancreas-smart-seq2.html).

**Claim boundary:** this is a real-data software demonstration. CIEDE2000, CVD simulations and affinity-fit scores are diagnostics, not proof of universal accessibility, biological similarity or improved human task performance.


## 1. Setup

The notebook does not silently modify the R environment. If dependencies are missing, set `INSTALL_MISSING <- TRUE` once and rerun this cell. When the notebook is launched inside this repository, the current scChromatic source is installed into a temporary session library so the demo does not depend on an older installed copy.


In [ ]:
INSTALL_MISSING <- FALSE
SEED <- 20260827L
K <- 15L
MIN_CELLS_PER_LABEL <- 30L
MIN_DONORS_PER_LABEL <- 2L

cran_packages <- c("BiocManager", "cli", "colorspace", "farver", "ggplot2", "jsonlite", "scales")
bioc_packages <- c(
  "scRNAseq", "SingleCellExperiment", "SummarizedExperiment",
  "scuttle", "scater", "BiocSingular"
)

missing_cran <- cran_packages[!vapply(cran_packages, requireNamespace, logical(1), quietly = TRUE)]
if (INSTALL_MISSING && length(missing_cran)) {
  install.packages(missing_cran, repos = "https://cloud.r-project.org")
}
if (INSTALL_MISSING) {
  missing_bioc <- bioc_packages[!vapply(bioc_packages, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing_bioc)) BiocManager::install(missing_bioc, ask = FALSE, update = FALSE)
}

missing <- c(cran_packages, bioc_packages)[
  !vapply(c(cran_packages, bioc_packages), requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing)) {
  stop(
    "Missing packages: ", paste(missing, collapse = ", "),
    ". Set INSTALL_MISSING <- TRUE and rerun this cell."
  )
}

find_repo_root <- function(path = getwd()) {
  repeat {
    description <- file.path(path, "DESCRIPTION")
    if (file.exists(description) && any(readLines(description, warn = FALSE) == "Package: scChromatic")) {
      return(normalizePath(path))
    }
    parent <- dirname(path)
    if (identical(parent, path)) return(NA_character_)
    path <- parent
  }
}

repo_root <- find_repo_root()
if (!is.na(repo_root)) {
  demo_library <- file.path(tempdir(), "scchromatic-demo-library")
  dir.create(demo_library, recursive = TRUE, showWarnings = FALSE)
  .libPaths(c(demo_library, .libPaths()))
  status <- system2(
    file.path(R.home("bin"), "R"),
    c("CMD", "INSTALL", "--no-multiarch", "-l", shQuote(demo_library), shQuote(repo_root)),
    stdout = TRUE, stderr = TRUE
  )
  install_status <- attr(status, "status")
  if (!is.null(install_status) && install_status != 0L) {
    stop("Local scChromatic installation failed:\n", paste(status, collapse = "\n"))
  }
}
if (!requireNamespace("scChromatic", quietly = TRUE)) {
  stop("Install scChromatic or launch this notebook from the scChromatic repository.")
}

suppressPackageStartupMessages({
  library(scChromatic)
  library(ggplot2)
})
set.seed(SEED)
theme_set(theme_classic(base_size = 11))

data.frame(
  R = as.character(getRversion()),
  scChromatic = as.character(packageVersion("scChromatic")),
  scRNAseq = as.character(packageVersion("scRNAseq")),
  stringsAsFactors = FALSE
)


## 2. Download and inspect the public data

`scRNAseq` downloads the prepared public dataset through Bioconductor's data infrastructure and caches it locally. Field names are resolved defensively because historical versions of the object have used spaces in `colData` names.


In [ ]:
sce_raw <- scRNAseq::SegerstolpePancreasData(ensembl = FALSE)
cell_metadata <- as.data.frame(SummarizedExperiment::colData(sce_raw))

pick_field <- function(candidates, available) {
  hit <- candidates[candidates %in% available]
  if (length(hit)) hit[[1L]] else NA_character_
}

fields <- c(
  label = pick_field(c("cell type", "CellType", "cell_type"), names(cell_metadata)),
  condition = pick_field(c("disease", "Disease", "condition"), names(cell_metadata)),
  donor = pick_field(c("individual", "Donor", "donor", "sample"), names(cell_metadata)),
  quality = pick_field(c("single cell well quality", "Quality", "quality"), names(cell_metadata))
)
if (anyNA(fields[c("label", "condition", "donor")])) {
  stop("Could not resolve label, condition and donor fields. Available fields: ",
       paste(names(cell_metadata), collapse = ", "))
}

dataset_overview <- data.frame(
  accession = "E-MTAB-5061",
  genes = nrow(sce_raw),
  cells = ncol(sce_raw),
  donors = length(unique(cell_metadata[[fields[["donor"]]]])),
  conditions = length(unique(cell_metadata[[fields[["condition"]]]])),
  resolved_label_field = fields[["label"]],
  resolved_donor_field = fields[["donor"]],
  resolved_condition_field = fields[["condition"]],
  stringsAsFactors = FALSE
)
dataset_overview


## 3. Prepare labels, normalize counts and compute PCA

Ambiguous author labels are excluded. To keep relationship estimates interpretable, retained cell types must have at least 30 cells and occur in at least two donors. PCA is computed from log-normalized expression and is used for both display and quantitative neighborhoods; UMAP is deliberately not used as a distance metric.


In [ ]:
label_raw <- trimws(as.character(cell_metadata[[fields[["label"]]]]))
donor_raw <- trimws(as.character(cell_metadata[[fields[["donor"]]]]))
condition_raw <- trimws(as.character(cell_metadata[[fields[["condition"]]]]))

ambiguous <- is.na(label_raw) | !nzchar(label_raw) |
  grepl("unclear|co-expression|unclassified|not applicable|empty", label_raw, ignore.case = TRUE)
base_keep <- !ambiguous & !is.na(donor_raw) & nzchar(donor_raw) &
  !is.na(condition_raw) & nzchar(condition_raw)

cell_support <- table(label_raw[base_keep])
donor_support <- tapply(donor_raw[base_keep], label_raw[base_keep], function(x) length(unique(x)))
eligible_labels <- intersect(
  names(cell_support)[cell_support >= MIN_CELLS_PER_LABEL],
  names(donor_support)[donor_support >= MIN_DONORS_PER_LABEL]
)
keep <- base_keep & label_raw %in% eligible_labels
sce <- sce_raw[, keep]

cd <- as.data.frame(SummarizedExperiment::colData(sce))
label <- trimws(as.character(cd[[fields[["label"]]]]))
donor <- trimws(as.character(cd[[fields[["donor"]]]]))
condition_original <- trimws(as.character(cd[[fields[["condition"]]]]))
condition_lower <- tolower(condition_original)
condition <- ifelse(
  grepl("normal|healthy|non.?diabet", condition_lower), "Healthy",
  ifelse(grepl("diabet|t2d|type ii", condition_lower), "Type 2 diabetes", condition_original)
)

sce <- scuttle::logNormCounts(sce)
set.seed(SEED)
sce <- scater::runPCA(
  sce, exprs_values = "logcounts", ncomponents = 20L,
  ntop = min(2000L, nrow(sce)), BSPARAM = BiocSingular::ExactParam()
)
pca <- SingleCellExperiment::reducedDim(sce, "PCA")
colnames(pca) <- paste0("PC", seq_len(ncol(pca)))

row_metadata <- as.data.frame(SummarizedExperiment::rowData(sce))
symbol_field <- pick_field(c("symbol", "Symbol", "gene_symbol"), names(row_metadata))
gene_symbol <- if (is.na(symbol_field)) rownames(sce) else as.character(row_metadata[[symbol_field]])
ins_index <- which(!is.na(gene_symbol) & toupper(gene_symbol) == "INS")

lineage_for <- function(x) {
  y <- tolower(x)
  ifelse(
    grepl("alpha|beta|delta|gamma|epsilon", y), "Endocrine",
    ifelse(
      grepl("acinar|ductal", y), "Exocrine",
      ifelse(
        grepl("endothelial|mesenchymal", y), "Stromal",
        ifelse(grepl("macrophage|mast|immune", y), "Immune", "Other")
      )
    )
  )
}

meta <- data.frame(
  cell_id = colnames(sce),
  label = label,
  lineage = lineage_for(label),
  donor = donor,
  condition = condition,
  PC1 = pca[, 1L],
  PC2 = pca[, 2L],
  signed_PC1 = as.numeric(scale(pca[, 1L])),
  library_size = as.numeric(colSums(SummarizedExperiment::assay(sce, "counts"))),
  stringsAsFactors = FALSE
)
meta$INS <- if (length(ins_index)) {
  as.numeric(SummarizedExperiment::assay(sce, "logcounts")[ins_index[[1L]], ])
} else {
  NA_real_
}

stopifnot(
  nrow(meta) == ncol(sce),
  length(unique(meta$label)) >= 6L,
  length(unique(meta$donor)) >= 2L,
  length(unique(meta$condition)) >= 2L,
  all(is.finite(as.matrix(pca[, seq_len(min(10L, ncol(pca))), drop = FALSE])))
)


In [ ]:
prepared_overview <- data.frame(
  retained_cells = nrow(meta),
  retained_cell_types = length(unique(meta$label)),
  donors = length(unique(meta$donor)),
  conditions = paste(sort(unique(meta$condition)), collapse = " / "),
  PCA_dimensions = ncol(pca),
  stringsAsFactors = FALSE
)
prepared_overview

label_by_condition <- addmargins(table(meta$label, meta$condition))
label_by_condition


In [ ]:
print_grid <- function(plots, ncol = length(plots)) {
  nrow <- ceiling(length(plots) / ncol)
  grid::grid.newpage()
  layout <- grid::grid.layout(nrow, ncol)
  grid::pushViewport(grid::viewport(layout = layout))
  for (i in seq_along(plots)) {
    row <- ceiling(i / ncol)
    col <- i - (row - 1L) * ncol
    print(plots[[i]], vp = grid::viewport(layout.pos.row = row, layout.pos.col = col))
  }
  invisible(plots)
}

embedding_plot <- function(data, map, title, shape_values = NULL) {
  p <- ggplot(data, aes(PC1, PC2, color = label)) +
    geom_point(size = 0.55, alpha = 0.8) +
    scale_color_sc_map(map, drop = TRUE) +
    coord_equal() +
    labs(title = title, color = "Cell type") +
    theme(legend.position = "right", legend.key.height = grid::unit(0.35, "cm"))
  if (!is.null(shape_values)) {
    p <- p + aes(shape = label) +
      scale_shape_manual(values = shape_values, drop = TRUE) +
      guides(shape = "none")
  }
  p
}


## 4. Advantage 1 — stable identity across conditions and subsets

A single named `sc_color_map` is created from the full cell-type universe and reused everywhere. The faceted plot below uses the exact same label-to-color contract for all cells, healthy donors and T2D donors.


In [ ]:
cell_map <- sc_color_map(meta$label, palette = "chromatic")
full_colors <- as_named_colors(cell_map)
meta$label <- factor(meta$label, levels = names(full_colors))

panel_data <- rbind(
  transform(meta, panel = "All cells"),
  transform(meta, panel = condition)
)
panel_data$panel <- factor(
  panel_data$panel,
  levels = c("All cells", "Healthy", "Type 2 diabetes")
)

ggplot(panel_data, aes(PC1, PC2, color = label)) +
  geom_point(size = 0.45, alpha = 0.75) +
  scale_color_sc_map(cell_map, drop = TRUE) +
  facet_wrap(~panel, nrow = 1) +
  coord_equal() +
  labs(
    title = "One persistent map across full and disease-specific views",
    color = "Cell type"
  ) +
  theme(legend.position = "bottom", legend.box = "vertical")


In [ ]:
ordered_labels <- names(full_colors)
drop_label <- ordered_labels[[min(2L, length(ordered_labels))]]
controlled_subset <- setNames(
  list(which(as.character(meta$label) != drop_label)),
  paste0("Without ", drop_label)
)
views <- c(
  list("All cells" = seq_len(nrow(meta))),
  controlled_subset,
  setNames(split(seq_len(nrow(meta)), meta$condition), paste0("Condition: ", names(split(seq_len(nrow(meta)), meta$condition)))),
  setNames(split(seq_len(nrow(meta)), meta$donor), paste0("Donor: ", names(split(seq_len(nrow(meta)), meta$donor))))
)

persistence_results <- do.call(rbind, lapply(names(views), function(view_name) {
  retained <- ordered_labels[ordered_labels %in% unique(as.character(meta$label[views[[view_name]]]))]
  persistent_subset <- as_named_colors(cell_map[retained])
  positional_subset <- stats::setNames(
    sc_palette("chromatic", length(retained), extend = "error"), retained
  )
  data.frame(
    view = view_name,
    retained_labels = length(retained),
    method = c("persistent named map", "positional reapplication"),
    recolored_fraction = c(
      mean(full_colors[retained] != persistent_subset[retained]),
      mean(full_colors[retained] != positional_subset[retained])
    ),
    stringsAsFactors = FALSE
  )
}))

stopifnot(
  all(persistence_results$recolored_fraction[persistence_results$method == "persistent named map"] == 0),
  any(persistence_results$recolored_fraction[persistence_results$method == "positional reapplication"] > 0)
)

ggplot(persistence_results, aes(view, recolored_fraction, fill = method)) +
  geom_col(position = "dodge") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  labs(
    title = "Filtering can recolor positional palettes; persistent maps do not",
    x = NULL, y = "Cell types recolored", fill = NULL
  ) +
  theme(axis.text.x = element_text(angle = 55, hjust = 1))


## 5. Advantage 2 — relationship-aware colors with an explicit stability budget

PCA neighborhoods are constructed within each donor and then averaged with equal donor weight. The resulting affinity matrix asks locally related labels to receive closer—but still distinct—colors. `unobserved = "zero"` is explicit: pairs never co-observed in a donor are treated as unrelated and reported in provenance.

The context map starts from the persistent canonical map, locks alpha and beta labels when present, and can change at most 25% of other assignments. The optimizer's objective should not worsen; the independent affinity-distance rank correlation is shown as a descriptive check.


In [ ]:
relationship <- sc_relationship_from_knn(
  pca[, seq_len(min(10L, ncol(pca))), drop = FALSE],
  labels = as.character(meta$label),
  sample = meta$donor,
  k = K,
  aggregate = "mean",
  unobserved = "zero"
)

locks <- ordered_labels[tolower(ordered_labels) %in% c("alpha", "beta")]
stability_budget <- max(1L, ceiling(0.25 * length(ordered_labels)))
baseline_evaluation <- sc_relationship_map(
  relationship, canonical = cell_map, locked = locks,
  stability_budget = 0L, seed = SEED
)
context_map <- sc_relationship_map(
  relationship, canonical = cell_map, locked = locks,
  stability_budget = stability_budget, seed = SEED
)

pair_metrics <- function(map, method) {
  labels <- rownames(relationship)
  colors <- as_named_colors(map)[labels]
  lab <- farver::decode_colour(unname(colors), to = "lab")
  distance <- farver::compare_colour(lab, lab, from_space = "lab", method = "cie2000")
  index <- which(upper.tri(relationship), arr.ind = TRUE)
  data.frame(
    method = method,
    label1 = labels[index[, 1L]],
    label2 = labels[index[, 2L]],
    affinity = as.numeric(relationship[index]),
    color_distance = as.numeric(distance[index]),
    stringsAsFactors = FALSE
  )
}

relationship_pairs <- rbind(
  pair_metrics(cell_map, "Persistent canonical"),
  pair_metrics(context_map, "Context + stability budget")
)

fit_summary <- do.call(rbind, lapply(split(relationship_pairs, relationship_pairs$method), function(x) {
  data.frame(
    method = x$method[[1L]],
    inverse_affinity_spearman = cor(1 - x$affinity, x$color_distance, method = "spearman"),
    stringsAsFactors = FALSE
  )
}))
fit_summary$optimizer_objective <- c(
  baseline_evaluation$context$final_objective,
  context_map$context$final_objective
)[match(fit_summary$method, c("Persistent canonical", "Context + stability budget"))]
fit_summary

stopifnot(
  context_map$context$final_objective <= baseline_evaluation$context$final_objective + 1e-12,
  context_map$context$stability_budget_used <= stability_budget,
  all(as_named_colors(context_map)[locks] == full_colors[locks])
)

ggplot(relationship_pairs, aes(affinity, color_distance, color = method)) +
  geom_point(size = 1.8, alpha = 0.75) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 0.7) +
  labs(
    title = "Relationship fit on the same donor-balanced PCA affinity matrix",
    x = "Cell-type affinity", y = "CIEDE2000 color distance", color = NULL
  )


In [ ]:
print_grid(list(
  embedding_plot(meta, cell_map, "Persistent canonical map"),
  embedding_plot(meta, context_map, "Context-aware map with bounded changes")
), ncol = 2L)


## 6. Hierarchy, expression and signed continuous values

The author cell types are grouped into a lightweight one-parent hierarchy for display. The same dataset also supplies real INS expression. A centered PC1 score provides a signed continuous variable without inventing a disease-effect claim.


In [ ]:
hierarchy_table <- unique(meta[c("label", "lineage")])
hierarchy_table <- hierarchy_table[order(hierarchy_table$lineage, hierarchy_table$label), ]
hierarchy_map <- sc_hierarchy_map(
  parent = hierarchy_table$lineage,
  child = as.character(hierarchy_table$label),
  separation = "balanced"
)

expression_value <- if (all(is.na(meta$INS))) log10(meta$library_size + 1) else meta$INS
expression_title <- if (all(is.na(meta$INS))) "Library size (INS symbol unavailable)" else "INS log-normalized expression"

p_hierarchy <- embedding_plot(meta, hierarchy_map, "Hierarchy-aware cell-type colors")
p_expression <- ggplot(meta, aes(PC1, PC2, color = expression_value)) +
  geom_point(size = 0.55) +
  scale_color_sc_c("viridis") +
  coord_equal() +
  labs(title = expression_title, color = NULL)
p_signed <- ggplot(meta, aes(PC1, PC2, color = signed_PC1)) +
  geom_point(size = 0.55) +
  scale_color_sc_c("chromatic_balance", midpoint = 0) +
  coord_equal() +
  labs(title = "Signed centered PC1", color = NULL)

print_grid(list(p_hierarchy, p_expression, p_signed), ncol = 3L)


## 7. Accessibility diagnostics and redundant shape encoding

The audit identifies the closest pair under normal, deutan, protan and tritan simulations. Confusable pairs below CIEDE2000 = 8 receive different shapes without changing their colors. The threshold is a diagnostic policy for this demonstration, not a universal perceptual boundary.


In [ ]:
audit <- sc_palette_audit(
  context_map,
  cvd = c("none", "deutan", "protan", "tritan")
)
audit$vision[c("vision", "min_distance", "median_distance", "worst_pair")]

shape_pool <- c(16L, 17L, 15L, 18L, 8L, 3L, 0L, 1L, 2L, 4L, 5L, 6L, 7L, 9L, 10L)
encoding <- sc_redundant_encoding(
  context_map, channel = "shape", min_cie2000 = 8,
  shapes = shape_pool
)
shape_values <- setNames(encoding$shape, encoding$label)
conflicts <- attr(encoding, "conflicts")

data.frame(
  labels = nrow(encoding),
  confusable_pairs = nrow(conflicts),
  shape_groups_used = length(unique(encoding$encoding_group)),
  colors_changed = FALSE,
  stringsAsFactors = FALSE
)

embedding_plot(
  meta, context_map,
  "Same persistent colors plus redundant shapes for confusable pairs",
  shape_values = shape_values
)


## 8. Portability, provenance and final scoreboard

The final relationship-aware map is serialized to JSON, restored, and checked for exact assignment equality. The scoreboard keeps the guaranteed persistence result separate from diagnostic relationship and CVD summaries.


In [ ]:
map_file <- tempfile("segerstolpe-scChromatic-", fileext = ".json")
write_sc_color_map(context_map, map_file)
restored_map <- read_sc_color_map(map_file)
roundtrip_ok <- identical(as_named_colors(context_map), as_named_colors(restored_map))
stopifnot(roundtrip_ok)

relationship_context <- attr(relationship, "sc_context")
audit_canonical <- sc_palette_audit(cell_map)
audit_context <- sc_palette_audit(context_map)

scoreboard <- data.frame(
  result = c(
    "Persistent-map recoloring across every tested view",
    "Mean positional recoloring across tested views",
    "Baseline relationship objective",
    "Context-map relationship objective",
    "Canonical assignments changed",
    "Canonical worst simulated-CVD minimum CIEDE2000",
    "Context-map worst simulated-CVD minimum CIEDE2000",
    "Zero-support relationship pairs",
    "JSON round-trip exact"
  ),
  value = c(
    max(persistence_results$recolored_fraction[persistence_results$method == "persistent named map"]),
    mean(persistence_results$recolored_fraction[persistence_results$method == "positional reapplication"]),
    baseline_evaluation$context$final_objective,
    context_map$context$final_objective,
    context_map$context$stability_budget_used,
    min(audit_canonical$vision$min_distance[audit_canonical$vision$vision != "none"]),
    min(audit_context$vision$min_distance[audit_context$vision$vision != "none"]),
    sum(relationship_context$pair_support$sample_count == 0L),
    roundtrip_ok
  ),
  interpretation = c(
    "must be 0",
    "lower is better; positional baseline",
    "optimizer diagnostic",
    "must not exceed baseline",
    paste0("budget = ", stability_budget, "; locked labels unchanged"),
    "diagnostic baseline",
    "diagnostic, not a universal pass/fail threshold",
    "explicitly filled as zero and recorded",
    "colors and labels preserved"
  ),
  stringsAsFactors = FALSE
)
scoreboard

data.frame(
  context_method = relationship_context$method,
  cells = relationship_context$cell_count,
  donors = relationship_context$sample_count,
  dimensions = relationship_context$coordinate_dimensions,
  k = relationship_context$k_requested,
  input_md5 = relationship_context$input_md5,
  matrix_md5 = relationship_context$matrix_md5,
  context_md5 = relationship_context$context_md5,
  stringsAsFactors = FALSE
)


## Interpretation

The strongest demonstrated advantage is exact semantic stability: the persistent map recolors zero labels after donor, disease and controlled label-removal subsets, while positional palette reapplication can silently reassign colors. The relationship-aware result adds a bounded, provenance-recorded trade-off between canonical stability, affinity fit and worst-case simulated-CVD separation. Hierarchy, continuous scales, audits, redundant shapes and JSON round trips all reuse the same public dataset and the same label universe.

For a manuscript claim about human readability or task accuracy, follow this notebook with the preregistered participant study described in `scChromatic_validation.ipynb`; CVD simulation alone is not sufficient.
